# Data Collection (Scraping)

**Forensic question:** What data was collected, from where, and when?

**Input artifacts:**
- `data/articles.db` — canonical article store
- `data/authors_manifest.jsonl` — discovery output

**Output artifacts:**
- `data/articles.jsonl` — export mirror (when scrape completes export)

**Run metadata:** (auto-populated by first code cell)


In [ ]:
# | echo: false
import sys
from datetime import UTC, datetime

from IPython.display import Markdown, display

from forensics.config import settings
from forensics.utils.provenance import compute_config_hash, compute_corpus_hash

config_hash = compute_config_hash(settings)
corpus_hash = compute_corpus_hash(settings.db_path)
run_timestamp = datetime.now(UTC).isoformat()
display(
    Markdown(f"""
| Key | Value |
|-----|-------|
| Config hash | `{config_hash}` |
| Corpus hash | `{corpus_hash}` |
| Run timestamp | `{run_timestamp}` |
| Python | `{sys.version}` |
""")
)

### Methodology

Two-step collection: WordPress REST discovery for metadata, then HTML fetch and parser extraction into `clean_text` with `content_hash` and `scraped_at` provenance.


In [ ]:
import polars as pl
from IPython.display import display

from forensics.config import settings

uri = f"sqlite:///{settings.db_path}"
q = """
SELECT a.name as author, a.slug,
       COUNT(*) as n_articles,
       MIN(t.published_date) as first_pub,
       MAX(t.published_date) as last_pub
FROM articles t JOIN authors a ON a.id = t.author_id
GROUP BY a.name, a.slug
"""
try:
    summary = pl.read_database_uri(q, uri)
    display(summary)
except Exception as exc:
    print("DB summary skipped:", exc)

In [ ]:
import plotly.express as px
import polars as pl

from forensics.config import settings
from forensics.utils.charts import register_forensics_template

register_forensics_template()
uri = f"sqlite:///{settings.db_path}"
q = """
SELECT strftime('%Y-%m', t.published_date) AS ym, a.slug, COUNT(*) AS n
FROM articles t JOIN authors a ON a.id = t.author_id
GROUP BY ym, a.slug ORDER BY ym
"""
try:
    vol = pl.read_database_uri(q, uri)
    fig = px.bar(vol, x="ym", y="n", color="slug", title="Articles per month")
    fig.show()
except Exception as exc:
    print("Volume chart skipped:", exc)

**Summary finding:** This chapter documents collection mechanics and empirical coverage; see tables and charts above for per-author counts and cadence.
